# DATA 605 Homework 5: Spark

## Spark background

* What is Spark?
* Why is Spark a popular framework?
* What is Spark SQL and why does it exist?
* What is a Spark DataFrame, and why is it useful?

* Apache Spark is an open-source, distributed computing framework.
* It is designed to process and analyze very large datasets quickly by dividing the work across multiple machines.
* Spark is popular as:
  * It is very fast compared to Hadoop MapReduce as it uses RAM instead of disk.
  * It can easily scale to huge distributed clusters without much changes in the code.
  * It is very simple to use.
  * Native support for multiple programming languages.
  * Enables programming at a higher level of abstraction.

* Spark SQL is a module in Spark that allows to run querys using SQL syntax.
  * It optimizes the query and runs against individual RDDs.
  * It is natively built into Spark.
  * Works with Hive data.

* Spark Dataframe is similar to a Pandas DataFrame in Python or a SQL table.
  * It is built on topvof RDDs
  * Designed and optimized for distributed tabular computing at scale.
  * It can queried using regular SQL syntax.
  * They are easy to work with and handy at scale.
  * Optimizes operations on DataFrames for better performance.



In [ ]:
# install pyspark
!pip install pyspark

I'll be performing analysis on the Heart Disease Dataset to predict Heart Failures.

The objective is to perform data preprocessing using *Spark SQL* and build a machine learning model using *Spark MLLib* that can accurately predict whether a person is likely to experience heart failure.

I have downloaded the dataset from Kaggle.
* Link: https://www.kaggle.com/code/tanmay111999/heart-failure-prediction-cv-score-90-5-models/notebook

In [ ]:
# Create a Spark session - required to use Spark DataFrames, SQL, and ML features.
from pyspark.sql import SparkSession

#.builder - starts the configuration process.
#.appName - gives Spark job a name that will used in logs
#.getOrCreate - creates a new Spark session if one doesn't already exist.
spark = SparkSession.builder \
    .appName("DATA605 Spark App") \
    .getOrCreate()

In [ ]:
# check if the session was created correctly
print(spark.sparkContext.appName)

DATA605 Spark App


##Explore dataset using Spark

In [ ]:
# Load the dataset as a Dataframe using Spark
# file_path is the path the csv file
# header =True indicates that the first row contains column names
# inferSchema=True automatically detects datatypes of the columns
file_path = 'heart.csv'
df = spark.read.csv(file_path, header=True, inferSchema=True)
# print the first 5 rows of the dataset
df.show(5)

+---+---+-------------+---------+-----------+---------+----------+-----+--------------+-------+--------+------------+
|Age|Sex|ChestPainType|RestingBP|Cholesterol|FastingBS|RestingECG|MaxHR|ExerciseAngina|Oldpeak|ST_Slope|HeartDisease|
+---+---+-------------+---------+-----------+---------+----------+-----+--------------+-------+--------+------------+
| 40|  M|          ATA|      140|        289|        0|    Normal|  172|             N|    0.0|      Up|           0|
| 49|  F|          NAP|      160|        180|        0|    Normal|  156|             N|    1.0|    Flat|           1|
| 37|  M|          ATA|      130|        283|        0|        ST|   98|             N|    0.0|      Up|           0|
| 48|  F|          ASY|      138|        214|        0|    Normal|  108|             Y|    1.5|    Flat|           1|
| 54|  M|          NAP|      150|        195|        0|    Normal|  122|             N|    0.0|      Up|           0|
+---+---+-------------+---------+-----------+---------+-

In [ ]:
# check the structure and datatype of the dataset.
df.printSchema()

root
 |-- Age: integer (nullable = true)
 |-- Sex: string (nullable = true)
 |-- ChestPainType: string (nullable = true)
 |-- RestingBP: integer (nullable = true)
 |-- Cholesterol: integer (nullable = true)
 |-- FastingBS: integer (nullable = true)
 |-- RestingECG: string (nullable = true)
 |-- MaxHR: integer (nullable = true)
 |-- ExerciseAngina: string (nullable = true)
 |-- Oldpeak: double (nullable = true)
 |-- ST_Slope: string (nullable = true)
 |-- HeartDisease: integer (nullable = true)



* Columns which are integer or double are numerical
* Columns which are string are categorical

In [ ]:
# print the number of rows and columns in the DataFrame
print(f"Dataset has {df.count()} rows and {len(df.columns)} columns")

Dataset has 918 rows and 12 columns


In [ ]:
# Display statistical summary
df.describe().show()

+-------+------------------+----+-------------+------------------+------------------+-------------------+----------+------------------+--------------+------------------+--------+-------------------+
|summary|               Age| Sex|ChestPainType|         RestingBP|       Cholesterol|          FastingBS|RestingECG|             MaxHR|ExerciseAngina|           Oldpeak|ST_Slope|       HeartDisease|
+-------+------------------+----+-------------+------------------+------------------+-------------------+----------+------------------+--------------+------------------+--------+-------------------+
|  count|               918| 918|          918|               918|               918|                918|       918|               918|           918|               918|     918|                918|
|   mean|53.510893246187365|NULL|         NULL|132.39651416122004| 198.7995642701525|0.23311546840958605|      NULL|136.80936819172112|          NULL|0.8873638344226581|    NULL| 0.5533769063180828|
| std

In [ ]:
# check datatype of Age
df.schema['Age'].dataType

IntegerType()

In [ ]:
from pyspark.sql.functions import isnan, when, count, col
# check dupliacte rows
df.groupBy(df.columns).agg(count('*').alias('duplicates')).filter(col('duplicates') > 1).show()

+---+---+-------------+---------+-----------+---------+----------+-----+--------------+-------+--------+------------+----------+
|Age|Sex|ChestPainType|RestingBP|Cholesterol|FastingBS|RestingECG|MaxHR|ExerciseAngina|Oldpeak|ST_Slope|HeartDisease|duplicates|
+---+---+-------------+---------+-----------+---------+----------+-----+--------------+-------+--------+------------+----------+
+---+---+-------------+---------+-----------+---------+----------+-----+--------------+-------+--------+------------+----------+



* There are no duplicate rows in this dataset

In [ ]:
from pyspark.sql.functions import sum as _sum

# check for Null values
# _sum() function counts the total number of nulls per columns
# .isNull returns True for nulls
# .cast converts boolean to int
# .alias labels the result with the column name
df.select(
    [_sum(col(c) \
          .isNull() \
          .cast("int")) \
          .alias(c) \
          for c in df.columns]
    ).show()

+---+---+-------------+---------+-----------+---------+----------+-----+--------------+-------+--------+------------+
|Age|Sex|ChestPainType|RestingBP|Cholesterol|FastingBS|RestingECG|MaxHR|ExerciseAngina|Oldpeak|ST_Slope|HeartDisease|
+---+---+-------------+---------+-----------+---------+----------+-----+--------------+-------+--------+------------+
|  0|  0|            0|        0|          0|        0|         0|    0|             0|      0|       0|           0|
+---+---+-------------+---------+-----------+---------+----------+-----+--------------+-------+--------+------------+



* There are no missing values in this dataset.

In [ ]:
# Check the unique values in each column
# Get the list of columns in the DataFrame
cols = df.columns

# Loop through each column to group by and count unique values
for col in cols:
  # check if the column has categorical values
  if df.schema[col].dataType.simpleString() == 'string':
    # groups the DataFrame by the column and count how many times each value appears
    df.groupBy(col).count().show()

+---+-----+
|Sex|count|
+---+-----+
|  F|  193|
|  M|  725|
+---+-----+

+-------------+-----+
|ChestPainType|count|
+-------------+-----+
|          NAP|  203|
|          ATA|  173|
|           TA|   46|
|          ASY|  496|
+-------------+-----+

+----------+-----+
|RestingECG|count|
+----------+-----+
|       LVH|  188|
|    Normal|  552|
|        ST|  178|
+----------+-----+

+--------------+-----+
|ExerciseAngina|count|
+--------------+-----+
|             Y|  371|
|             N|  547|
+--------------+-----+

+--------+-----+
|ST_Slope|count|
+--------+-----+
|    Flat|  460|
|      Up|  395|
|    Down|   63|
+--------+-----+



### Data Quality Check
* I checked the dataset for duplicate rows, missing values, and inconsistencies in categorical features.
* I found that the dataset is clean
    
  — it contains no duplicate records, no null values, and all categorical columns have correct, typo-free values.
* Since the data is well-structured, I don't have to perform any additional modifications. The dataset is ready for modeling.

### Check for Outliers

In [ ]:
from pyspark.sql.functions import col, mean, stddev, abs, max, min

# Function to check for outliers using Z-score method
def check_outliers(column):
    print(f"\nCheck outliers in column: {column}")

    # Calculate mean and standard deviation
    stats = df.agg(
        mean(col(column)).alias('mean'),
        stddev(col(column)).alias('stddev'),
        max(col(column)).alias('max'),
        min(col(column)).alias('min')
    ).collect()[0]

    # Skip column if stddev is 0
    if stats.stddev == 0:
        print("Cannot compute Z-score as stddev = 0. Skipping this column.")
        return

    # Compute Z-score: Z = (x–μ)/σ
    df_zscored = df.withColumn(f'z_score_{column}', (col(column) - stats.mean) / stats.stddev)

    # Filter rows where Z-score is greater than 3 or less than -3
    outliers_df = df_zscored.filter(abs(col(f'z_score_{column}')) > 3)

    # Results
    print(f'Max: {stats.max}\nMin: {stats.min}\nMean: {stats.mean}')
    print(f'Standard Deviation: {stats.stddev}')
    print('\nDataFrame with Z-scores added:')
    df_zscored.show(5)

    outlier_count = outliers_df.count()
    print(f'\nThere are {outlier_count} outliers:')

    if outlier_count > 0:
        outliers_df.show()

    print('='*100)

# Get numeric columns
numeric_columns = [c for c in df.columns if df.schema[c].dataType.simpleString() in ['int', 'double']]

# don't check for the last column - HeartDisease
for _ in numeric_columns[:-1]:
    check_outliers(_)


Check outliers in column: Age
Max: 77
Min: 28
Mean: 53.510893246187365
Standard Deviation: 9.43261650673202

DataFrame with Z-scores added:
+---+---+-------------+---------+-----------+---------+----------+-----+--------------+-------+--------+------------+-------------------+
|Age|Sex|ChestPainType|RestingBP|Cholesterol|FastingBS|RestingECG|MaxHR|ExerciseAngina|Oldpeak|ST_Slope|HeartDisease|        z_score_Age|
+---+---+-------------+---------+-----------+---------+----------+-----+--------------+-------+--------+------------+-------------------+
| 40|  M|          ATA|      140|        289|        0|    Normal|  172|             N|    0.0|      Up|           0|-1.4323590105189474|
| 49|  F|          NAP|      160|        180|        0|    Normal|  156|             N|    1.0|    Flat|           1| -0.478222902729901|
| 37|  M|          ATA|      130|        283|        0|        ST|   98|             N|    0.0|      Up|           0|-1.7504043797819628|
| 48|  F|          ASY|      13

* Outliers were detected in the columns RestingBP, Cholesterol, FastingBS, MaxHR, and Oldpeak, indicating the presence of extreme clinical values.
These correspond to rare and medically abnormal cases.

* Interestingly, many of these outlier instances are associated with a target label of HeartDisease = 1, suggesting that patients exhibiting extreme physiological measurements are more likely to have heart disease.

* This observation aligns with medical expectations and supports the relevance of these features in predicting heart failure outcomes

## Machine Learing

In [ ]:
df.dtypes

[('Age', 'int'),
 ('Sex', 'string'),
 ('ChestPainType', 'string'),
 ('RestingBP', 'int'),
 ('Cholesterol', 'int'),
 ('FastingBS', 'int'),
 ('RestingECG', 'string'),
 ('MaxHR', 'int'),
 ('ExerciseAngina', 'string'),
 ('Oldpeak', 'double'),
 ('ST_Slope', 'string'),
 ('HeartDisease', 'int')]

In [ ]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml import Pipeline
from pyspark.sql.functions import col

In [ ]:
# Encode categorical features
categorical_cols = [c for c, t in df.dtypes if t == 'string']
indexers = [StringIndexer(inputCol=c, outputCol=c+"_index", handleInvalid="keep") for c in categorical_cols]
encoders = [OneHotEncoder(inputCol=c+"_index", outputCol=c+"_vec") for c in categorical_cols]

* StringIndexer and OneHotEncoder are built in functions in pyspark.ml.feature

* **StringIndexer:**
  * converts strings to numeric indexes
  * inputCol is the input column
  * outputCol is the new name of the column
  * handleInvalid = "keep" assigns unseen or null values to a default index
  * Ex: Values RestingECG are LVH, Normal, ST. They maybe converted as:
      * LVH - 0
      * Normal - 1
      * ST - 2

* **OneHotEncoder:**
  * This takes the indexed columns and applies one-hot encoding to them.
  * One-hot encoding turns the category index into a binary vector.
  * inputCol is the input column
  * outputCol is the new name of the column


In [ ]:
# Assemble features into one vector
numeric_cols = [c for c, t in df.dtypes if t in ['int', 'double'] and c != 'HeartDisease']
assembler_inputs = [c+"_vec" for c in categorical_cols] + numeric_cols
assembler = VectorAssembler(inputCols=assembler_inputs, outputCol="features")

**assembler_inputs** builds a list of all input features:

  * One-hot encoded vectors
  * All numeric columns

**VectorAssembler** mergers all the input features into a single feature variable. This is needed to pass it as an input the machine learning algorithm.

In [ ]:
# Initialize Logistic Regression
lr = LogisticRegression(labelCol="HeartDisease", featuresCol="features")

# Create pipeline
pipeline = Pipeline(stages=indexers + encoders + [assembler, lr])

# Train-test split (80-20)
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

# Fit the model
model = pipeline.fit(train_df)

# Make predictions
predictions = model.transform(test_df)

* We use the **logistic regression** model from Spark MLLib.
  * labelCol="HeartDisease" is the the target variable to be predicted.
  * featuresCol="features" is the column that contains all input data from VectorAssembler.

* The **Pipeline** creates a pipeline to combines all transformation and model steps

  * indexers -> convert string -> indexing
  * encoders -> convert indexing -> one-hot vectors
  * assembler -> combine all features
  * lr -> logistic regression model

* **randomSplit** function splits the original dataset into 80% for training and 20% for testing.

* seed=42 ensures the split is reproducible.

* **pipeline.fit()** automatically applies all data transformations and fits the training dataset into logistic regression model.

* **model.transform()** applies the full pipeline to the test dataset.

In [ ]:
# Evaluate using AUC (Area Under ROC Curve)
evaluator = BinaryClassificationEvaluator(labelCol="HeartDisease", metricName="areaUnderROC")
auc = evaluator.evaluate(predictions)

print(f"Area Under ROC (AUC): {auc:.4f}")

Area Under ROC (AUC): 0.9110


In [ ]:
# Confusion matrix
conf_matrix = predictions.groupBy("HeartDisease", "prediction").count().orderBy("HeartDisease", "prediction")
conf_matrix.show()

+------------+----------+-----+
|HeartDisease|prediction|count|
+------------+----------+-----+
|           0|       0.0|   49|
|           0|       1.0|   12|
|           1|       0.0|    8|
|           1|       1.0|   80|
+------------+----------+-----+



In [ ]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# List of metrics to evaluate
metrics = ["accuracy", "weightedPrecision", "weightedRecall", "f1"]
results = {}

# Loop through each metric and evaluate
for metric in metrics:
    evaluator = MulticlassClassificationEvaluator(
        labelCol="HeartDisease", predictionCol="prediction", metricName=metric
    )
    results[metric] = evaluator.evaluate(predictions)

# Print results
print("Model Evaluation Metrics:")
for metric in metrics:
    print(f"{metric}: {results[metric]:.4f}")


Model Evaluation Metrics:
accuracy: 0.8658
weightedPrecision: 0.8655
weightedRecall: 0.8658
f1: 0.8650


* I have chosen a binary classification task, where the goal is to predict if a person is at risk of heart disease (1) or not (0) based on clinical features
* I have trained the data on Logistic Regression Algorithm as it is best used for binary classification problems.
* The algorithm works by modeling the probability that a given input belongs to the positive class using a sigmoid function, and then applies a threshold of 0.5 to make the final prediction (0 or 1).
* Logistic Regression also provides probability outputs, which can be useful for risk scoring and medical decision-making.

I evaluated the model using several performance metrics: Accuracy, Precision, Recall, F1 Score, and AUC (Area Under the ROC Curve).

These metrics give a comprehensive view of how well the model is performing:
* Accuracy measures the overall correctness of predictions.
* Precision tells how many of the patients predicted to have heart disease actually had it.
* Recall indicates how many true heart disease cases the model was able to detect.
* F1 Score balances both precision and recall into a single metric.
* AUC reflects the model’s ability to distinguish between patients with and without heart disease across all thresholds — a higher AUC means better ranking of patients by risk.

The model achieved an accuracy of 86.58% and an AUC of 0.9110, indicating strong performance in identifying patients at risk. The results suggest that the logistic regression model is a good fit for this binary classification task, with well-balanced sensitivity and specificity.

In [ ]:
! pip install graphframes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 4.6 MB/s eta 0:00:00
